# EduPredict: Machine Learning & Exploratory Data Analysis (EDA)
### Environment: Jupyter Anaconda Notebook 3 (Python 3.10+)
**Objective**: Data exploration, cohort feature engineering, and predictive modeling for student retention and academic risk classification.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Set seed for reproducibility
np.random.seed(42)
print("[✓] Core data science packages imported successfully.")

## 1. Load / Simulate Longitudinal Student Cohort Data
Simulating a representative cohort of 1,500 students with engagement, LMS telemetry, and exam features.

In [ ]:
n_samples = 1500

student_ids = [f"STU-2026-{i:04d}" for i in range(1, n_samples + 1)]
departments = np.random.choice(["Computer Science", "Data Science", "Software Engineering", "Business Analytics"], size=n_samples)
attendance_rate = np.clip(np.random.normal(loc=78, scale=14, size=n_samples), 20, 100)
weekly_lms_hours = np.clip(np.random.normal(loc=8.2, scale=3.8, size=n_samples), 0.5, 25.0)
midterms_avg = np.clip(0.42 * attendance_rate + 0.3 * (weekly_lms_hours * 3.5) + np.random.normal(20, 10, size=n_samples), 20, 100)
assignments_score = np.clip(midterms_avg + np.random.normal(5, 7, size=n_samples), 15, 100)
consecutive_absences = np.clip(np.random.exponential(scale=1.1, size=n_samples).astype(int), 0, 8)
prior_gpa = np.clip(np.random.normal(loc=3.1, scale=0.5, size=n_samples), 1.5, 4.0)

# Latent probability of academic distress
z = (-0.075 * attendance_rate 
     - 0.110 * weekly_lms_hours 
     - 0.045 * midterms_avg 
     + 0.320 * consecutive_absences 
     - 0.750 * prior_gpa 
     + 3.95)
dropout_probs = 1.0 / (1.0 + np.exp(-z))
is_dropout = (np.random.rand(n_samples) < dropout_probs).astype(int)

df = pd.DataFrame({
    "student_id": student_ids,
    "department": departments,
    "attendance_rate": np.round(attendance_rate, 2),
    "weekly_lms_hours": np.round(weekly_lms_hours, 2),
    "midterms_avg": np.round(midterms_avg, 2),
    "assignments_score": np.round(assignments_score, 2),
    "consecutive_absences": consecutive_absences,
    "prior_gpa": np.round(prior_gpa, 2),
    "is_at_risk": is_dropout
})

print(f"Cohort size: {df.shape[0]} rows, At-risk rate: {df['is_at_risk'].mean():.2%}")
df.head()

## 2. Exploratory Data Analysis (EDA)
Checking feature correlations against student retention risk.

In [ ]:
correlations = df[["attendance_rate", "weekly_lms_hours", "midterms_avg", "assignments_score", "consecutive_absences", "prior_gpa", "is_at_risk"]].corr()
print("Correlation with 'is_at_risk':")
print(correlations["is_at_risk"].sort_values(ascending=False))

## 3. Train-Test Split and Model Training
Training a Logistic Regression and a Random Forest Classifier.

In [ ]:
features = ["attendance_rate", "weekly_lms_hours", "midterms_avg", "assignments_score", "consecutive_absences", "prior_gpa"]
X = df[features]
y = df["is_at_risk"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 1. Logistic Regression Model
log_model = LogisticRegression(random_state=42)
log_model.fit(X_train_scaled, y_train)

# 2. Random Forest Model
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

print("[✓] Both models fitted successfully.")

## 4. Model Evaluation & ROC Metrics

In [ ]:
y_pred_log = log_model.predict(X_test_scaled)
y_prob_log = log_model.predict_proba(X_test_scaled)[:, 1]
auc_score = roc_auc_score(y_test, y_prob_log)

print(f"=== Logistic Regression ROC-AUC: {auc_score:.4f} ===")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_log, target_names=["Retained", "At-Risk"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))

## 5. Feature Importances
Examining the dominant drivers influencing student retention predictions.

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)
print("Random Forest Feature Importances:")
for feat, imp in importances.items():
    print(f"  - {feat:22s}: {imp * 100:.2f}%")